# Vocabulary Parallel Embedding

## Overview

Split large vocabulary embeddings across GPUs for memory efficiency.

### Use Case
- Large vocabularies (100K+ tokens)
- Large embedding dimensions
- Memory-constrained training

In [ ]:
import torch
import torch.nn as nn

class VocabParallelEmbedding(nn.Module):
    """Embedding layer with vocabulary parallelism."""
    
    def __init__(self, vocab_size, embed_dim, tp_size, tp_rank):
        super().__init__()
        self.vocab_size = vocab_size
        self.tp_size = tp_size
        self.tp_rank = tp_rank
        
        # Each GPU handles vocab_size/tp_size tokens
        self.vocab_per_gpu = vocab_size // tp_size
        self.vocab_start = tp_rank * self.vocab_per_gpu
        self.vocab_end = self.vocab_start + self.vocab_per_gpu
        
        self.embedding = nn.Embedding(self.vocab_per_gpu, embed_dim)
    
    def forward(self, input_ids):
        # Mask tokens not in this GPU's range
        mask = (input_ids >= self.vocab_start) & (input_ids < self.vocab_end)
        local_ids = (input_ids - self.vocab_start).clamp(0, self.vocab_per_gpu - 1)
        
        output = self.embedding(local_ids)
        output = output * mask.unsqueeze(-1).float()
        
        # AllReduce to combine results
        return output  # Would call dist.all_reduce here

# Example: 100K vocab split across 8 GPUs
embed = VocabParallelEmbedding(100000, 4096, tp_size=8, tp_rank=0)
print(f"Vocab range: {embed.vocab_start} - {embed.vocab_end}")

## Memory Savings

| Vocab Size | Embed Dim | Full (GB) | Per GPU (8-way) |
|------------|-----------|-----------|----------------|
| 50K | 4096 | 0.8 | 0.1 |
| 100K | 4096 | 1.6 | 0.2 |
| 250K | 8192 | 8.0 | 1.0 |